#BERT MODEL SENTIMENT ANALYSIS

In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
!pip install transformers datasets evaluate accelerate -q

In [ ]:
#IMPORTING THE PACKAGES
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from transformers import DataCollatorWithPadding
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [ ]:
import torch
from datasets import load_dataset
from transformers import ( AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer)
import evaluate
import numpy as np

In [ ]:
dataset = load_dataset("imdb")

print(dataset)

In [ ]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
#TOKENIZATION FUNCTION
def tokenize_function(example):
    return tokenizer( example["text"], padding="max_length", truncation=True, max_length=256)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

In [ ]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42)
test_dataset  = tokenized_dataset["test"].shuffle(seed=42)

In [ ]:
#MODEL LOADING
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using: {device}")

In [ ]:
import numpy as np
#DEFINING METRICS
from sklearn.metrics import ( accuracy_score,precision_score,recall_score,f1_score)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='binary', zero_division=0)
    recall = recall_score(labels, predictions, average='binary', zero_division=0)
    f1 = f1_score(labels, predictions, average='binary', zero_division=0)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
#TRAINING ARGUMENTS
training_args = TrainingArguments(
    output_dir ="./results",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    logging_steps = 500,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_accuracy",
    greater_is_better = True,
    report_to = "none"
)

In [ ]:
#DATA COLLATOR
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
#TRAINING MODEL
trainer.train()

In [ ]:
#EVALUATION AND RESULTS
results = trainer.evaluate()
for key, value in results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#SAVING THE TRAINED BERT MODEL AND TOKENIZER
import json

save_path = "/content/drive/MyDrive/BERT_Model_Final_Sentiment_Analysis"

# Save model and tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save evaluation results
results_path = f"{save_path}/results.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Model, tokenizer, and results saved successfully!")